In [ ]:
"""
AgriLST-ML — AOI / Farm-Level Inference & GeoTIFF Export  (notebook version)
=============================================================================
Runs the trained, district-calibrated XGBoost AgriLST-ML production model
over a user-supplied AOI (farm shapefile, or a lat/lon point + buffer) and
exports two GeoTIFFs, ready to style manually in QGIS:

    1. LST_<name>_<date>.tif    — predicted Land Surface Temperature, °C, 10 m
    2. NDVI_<name>_<date>.tif   — Sentinel-2 NDVI, 10 m, same AOI + date

This version is written to be pasted into Colab cells directly — no argparse,
no CLI. Everything you need to set is in the CONFIG block in STEP 1 below.

-----------------------------------------------------------------------------
HOW TO RUN (in order)
-----------------------------------------------------------------------------
STEP 0  Install deps (once):
            !pip install -q earthengine-api geemap rasterio geopandas shapely joblib xgboost
STEP 1  Edit the CONFIG block below: paths, AOI (shapefile OR lat/lon+buffer),
        target date, output name.
STEP 2  Run print_available_districts() — it loads your model's metadata and
        prints every district name it was trained/calibrated on. Copy the
        exact string for the district your AOI sits in into DISTRICT below.
        Farm-level AOIs sit inside one district, so this is a single manual
        choice rather than an automatic (and error-prone) spatial join.
STEP 3  Run run_prediction() — builds the feature stack, downloads it,
        predicts, clips to your exact AOI polygon, writes the two GeoTIFFs.

-----------------------------------------------------------------------------
⚠️  TRAIN/SERVE CONSISTENCY — read before trusting output
-----------------------------------------------------------------------------
Feature engineering (airtemp_C_squared, NDVI_x_AirTemp, NDWI_div_Clay,
coastline_distance_proxy) is copied verbatim from your predict_lst().
The raw GEE sources in CONFIG (Sentinel-2, ERA5, SRTM, SoilGrids/OpenLandMap,
CHIRPS) are reconstructed from predict_lst()'s docstring — I don't have
Data_Preparation_Pipeline v9, so band names / cloud-masking / ERA5
aggregation window are my best reconstruction, not a verified match. The
script will run and produce numbers regardless of whether these match; it
can't detect a mismatch itself. Run the SANITY CHECK note at the bottom
before trusting a real AOI.
-----------------------------------------------------------------------------
"""

import os
import datetime as dt

import numpy as np
import pandas as pd
import joblib

# ─────────────────────────────────────────────────────────────────────────
# STEP 1 — CONFIG: edit everything in this block
# ─────────────────────────────────────────────────────────────────────────

# --- model artifacts (already saved by your training notebook) ---
MODEL_PATH = 'FINAL_MODEL/agrilst_final_xgboost.pkl'
META_PATH  = 'FINAL_MODEL/agrilst_final_metadata.pkl'
OUTPUT_DIR = 'Output'

EE_PROJECT = 'Your-project-ID'    # <-- REQUIRED: your GEE cloud project

# --- AOI: fill in ONE of these two modes, leave the other at None/0 ---
SHAPEFILE_PATH = "farms_shapefile.shp"                   # e.g. '/content/drive/MyDrive/farms/farm1.shp'
SHAPEFILE_BUFFER_M = 100                  # optional extra margin around the polygon, metres

POINT_LAT = None                       # e.g. 20.7453
POINT_LON = None                      # e.g. 78.6022
POINT_BUFFER_M = 1000                    # required for point mode: radius, metres

# --- date & naming ---
TARGET_DATE = '2025-12-15'              # YYYY-MM-DD
AOI_NAME    = 'farm1'                   # used in output filenames

# --- district (manual — see STEP 2 / print_available_districts()) ---
DISTRICT = "Satara"                         # e.g. 'Wardha'  -- copy exact string from the printed list

# --- GEE asset / collection IDs — verify against Data_Preparation_Pipeline v9 ---
S2_COLLECTION      = 'COPERNICUS/S2_SR_HARMONIZED'
ERA5_COLLECTION    = 'ECMWF/ERA5_LAND/DAILY_AGGR'     # band 'temperature_2m' (Kelvin, daily mean)
CHIRPS_COLLECTION  = 'UCSB-CHG/CHIRPS/DAILY'          # band 'precipitation' (mm/day)
SRTM_ASSET         = 'USGS/SRTMGL1_003'               # band 'elevation'
SOILGRIDS_CLAY     = 'projects/soilgrids-isric/clay_mean'   # band 'clay_0-5cm_mean' (g/kg)
SOILGRIDS_SAND     = 'projects/soilgrids-isric/sand_mean'   # band 'sand_0-5cm_mean'
SOIL_TEXTURE_ASSET = 'OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02'  # band 'b0', USDA class 1-12

CLOUD_SEARCH_DAYS = 15      # +/- window around target date for a cloud-free S2 composite
MAX_CLOUD_PCT     = 40
EXPORT_SCALE_M    = 10
EXPORT_CRS        = 'EPSG:32643'   # UTM 43N — most of Maharashtra. Use EPSG:32642 for the western edge.


# ─────────────────────────────────────────────────────────────────────────
# EARTH ENGINE SETUP
# ─────────────────────────────────────────────────────────────────────────

def init_ee():
    import ee
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    return ee


# ─────────────────────────────────────────────────────────────────────────
# STEP 2 — district lookup: run this first and copy a name into DISTRICT
# ─────────────────────────────────────────────────────────────────────────

def print_available_districts(meta_path=META_PATH):
    meta = joblib.load(meta_path)
    names = sorted(meta['district_bias_correction'].keys())
    print(f"{len(names)} districts available for calibration — copy the exact")
    print("string for your AOI's district into DISTRICT in the CONFIG block:\n")
    for n in names:
        print(f"  '{n}'")
    return names


# ─────────────────────────────────────────────────────────────────────────
# AOI HANDLING — shapefile OR lat/lon + buffer
# ─────────────────────────────────────────────────────────────────────────

def load_aoi(ee):
    import geopandas as gpd
    from shapely.geometry import Point
    import json

    if SHAPEFILE_PATH:
        gdf = gpd.read_file(SHAPEFILE_PATH)
        if gdf.crs is None:
            raise ValueError("Shapefile has no CRS defined — set one before running.")
        gdf = gdf.to_crs(epsg=4326)
        if SHAPEFILE_BUFFER_M > 0:
            gdf_m = gdf.to_crs(epsg=32643)
            gdf_m['geometry'] = gdf_m.buffer(SHAPEFILE_BUFFER_M)
            gdf = gdf_m.to_crs(epsg=4326)
        geom_local = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
        geojson = json.loads(gpd.GeoSeries([geom_local]).to_json())['features'][0]['geometry']
        return ee.Geometry(geojson), geom_local

    elif POINT_LAT is not None and POINT_LON is not None:
        if not POINT_BUFFER_M or POINT_BUFFER_M <= 0:
            raise ValueError("Point AOI requires POINT_BUFFER_M > 0 (radius in metres).")
        pt = Point(POINT_LON, POINT_LAT)
        gdf = gpd.GeoDataFrame({'geometry': [pt]}, crs='EPSG:4326')
        gdf_m = gdf.to_crs(epsg=32643)
        gdf_m['geometry'] = gdf_m.buffer(POINT_BUFFER_M)
        gdf = gdf_m.to_crs(epsg=4326)
        geom_local = gdf.geometry.iloc[0]
        ee_geom = ee.Geometry.Point([POINT_LON, POINT_LAT]).buffer(POINT_BUFFER_M)
        return ee_geom, geom_local

    else:
        raise ValueError("Set either SHAPEFILE_PATH, or both POINT_LAT and POINT_LON, in CONFIG.")


# ─────────────────────────────────────────────────────────────────────────
# GEE FEATURE STACK — mirrors the training feature set
# ─────────────────────────────────────────────────────────────────────────

FEATURE_BAND_ORDER = [
    'ndvi', 'ndwi', 'airtemp_C', 'airtemp_C_squared', 'doy_sin', 'doy_cos',
    'elevation', 'soil_clay', 'soil_sand', 'rain_0d', 'rain_15d',
    'district_mean_elevation', 'coastline_distance_proxy',
    'NDVI_x_AirTemp', 'NDWI_div_Clay', 'soil_texture',
]


def mask_s2_clouds(ee, img):
    scl = img.select('SCL')
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
    return img.updateMask(bad.Not())


def build_feature_image(ee, aoi_geom, target_date_str, district_elev_value):
    """
    Builds a single multiband ee.Image, one band per model feature.
    district_elev_value: scalar (meta['district_elev_map'][DISTRICT], or the
    AOI's own SRTM elevation if DISTRICT is unset/unknown — same fallback
    predict_lst() uses).
    """
    target_date = ee.Date(target_date_str)
    doy = dt.datetime.strptime(target_date_str, '%Y-%m-%d').timetuple().tm_yday
    angle = 2 * np.pi * doy / 365.25
    doy_sin_val, doy_cos_val = float(np.sin(angle)), float(np.cos(angle))

    s2 = (ee.ImageCollection(S2_COLLECTION)
          .filterBounds(aoi_geom)
          .filterDate(target_date.advance(-CLOUD_SEARCH_DAYS, 'day'),
                      target_date.advance(CLOUD_SEARCH_DAYS + 1, 'day'))
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
          .map(lambda im: mask_s2_clouds(ee, im)))
    s2_median = s2.median()
    ndvi = s2_median.normalizedDifference(['B8', 'B4']).rename('ndvi')
    ndwi = s2_median.normalizedDifference(['B3', 'B8']).rename('ndwi')  # McFeeters NDWI

    era5 = (ee.ImageCollection(ERA5_COLLECTION)
            .filterDate(target_date, target_date.advance(1, 'day'))
            .first())
    airtemp_C = era5.select('temperature_2m').subtract(273.15).rename('airtemp_C')
    airtemp_C_sq = airtemp_C.pow(2).rename('airtemp_C_squared')

    chirps = ee.ImageCollection(CHIRPS_COLLECTION)
    rain_0d = chirps.filterDate(target_date, target_date.advance(1, 'day')).sum().rename('rain_0d')
    rain_15d = chirps.filterDate(target_date.advance(-14, 'day'), target_date.advance(1, 'day')) \
        .sum().rename('rain_15d')

    elevation = ee.Image(SRTM_ASSET).select('elevation').rename('elevation')
    soil_clay = ee.Image(SOILGRIDS_CLAY).select(0).divide(10).rename('soil_clay')   # g/kg -> %
    soil_sand = ee.Image(SOILGRIDS_SAND).select(0).divide(10).rename('soil_sand')
    soil_texture = ee.Image(SOIL_TEXTURE_ASSET).select('b0').rename('soil_texture')

    dist_elev_img = ee.Image.constant(float(district_elev_value)).rename('district_mean_elevation')

    doy_sin_img = ee.Image.constant(doy_sin_val).rename('doy_sin')
    doy_cos_img = ee.Image.constant(doy_cos_val).rename('doy_cos')

    coastline_proxy = elevation.multiply(0.1).add(soil_sand.multiply(0.05)) \
        .rename('coastline_distance_proxy')
    ndvi_x_airtemp = ndvi.multiply(airtemp_C).rename('NDVI_x_AirTemp')
    ndwi_div_clay = ndwi.divide(soil_clay.add(1.0)).rename('NDWI_div_Clay')

    stack = ee.Image.cat([
        ndvi, ndwi, airtemp_C, airtemp_C_sq, doy_sin_img, doy_cos_img,
        elevation, soil_clay, soil_sand, rain_0d, rain_15d,
        dist_elev_img, coastline_proxy, ndvi_x_airtemp, ndwi_div_clay,
        soil_texture,
    ]).clip(aoi_geom)

    return stack, ndvi.clip(aoi_geom)


# ─────────────────────────────────────────────────────────────────────────
# EXPORT TO LOCAL GEOTIFF, RUN MODEL, WRITE RESULTS
# ─────────────────────────────────────────────────────────────────────────

def export_image_locally(image, aoi_geom, out_path, band_names, scale=EXPORT_SCALE_M):
    import geemap
    geemap.ee_export_image(
        image.select(band_names),
        filename=out_path,
        scale=scale,
        region=aoi_geom,
        crs=EXPORT_CRS,
        file_per_band=False,
    )
    return out_path


def reproject_geom(geom, dst_crs, src_crs='EPSG:4326'):
    """aoi_local (from load_aoi) is always WGS84 lon/lat. GeoTIFFs exported
    from GEE are in EXPORT_CRS (metres). rasterio.mask needs the mask
    geometry in the SAME CRS as the raster being masked, or you get exactly
    the 'Input shapes do not overlap raster' error — the shape isn't
    actually outside the raster, it's just being read in the wrong units."""
    from pyproj import Transformer
    from shapely.ops import transform as shp_transform
    transformer = Transformer.from_crs(src_crs, dst_crs, always_xy=True)
    return shp_transform(transformer.transform, geom)


def run_inference(stack_tif_path, ndvi_tif_path, meta, model, bias_correction,
                   aoi_local_geom, out_dir, name, date_str):
    import rasterio
    from rasterio.mask import mask as rio_mask

    with rasterio.open(stack_tif_path) as src:
        profile = src.profile.copy()
        arr = src.read()  # (n_feat, H, W), band order = FEATURE_BAND_ORDER
        nodata_mask = np.any(~np.isfinite(arr), axis=0)

    n_feat = len(FEATURE_BAND_ORDER)
    H, W = arr.shape[1], arr.shape[2]
    flat = arr.reshape(n_feat, -1).T  # (H*W, n_feat)
    df = pd.DataFrame(flat, columns=FEATURE_BAND_ORDER)

    training_categories = meta.get('soil_texture_categories', list(range(1, 13)))
    df['soil_texture'] = pd.Categorical(
        df['soil_texture'].round().astype('Int64'), categories=training_categories
    )

    raw_preds = model.predict(df[meta['all_features']])
    corrected = raw_preds + float(bias_correction)

    lst_2d = corrected.reshape(H, W).astype('float32')
    lst_2d[nodata_mask] = np.nan

    out_profile = profile.copy()
    out_profile.update(count=1, dtype='float32', nodata=np.nan)

    lst_out = os.path.join(out_dir, f'LST_{name}_{date_str}.tif')
    with rasterio.open(lst_out, 'w', **out_profile) as dst:
        dst.write(lst_2d, 1)
        dst.set_band_description(1, 'LST_predicted_C')

    # clip precisely to the AOI polygon (trims corner pixels outside an
    # irregular farm boundary that the near-bbox GEE export region includes)
    with rasterio.open(lst_out) as src:
        aoi_in_raster_crs = reproject_geom(aoi_local_geom, src.crs)
        clipped, transform = rio_mask(src, [aoi_in_raster_crs.__geo_interface__], crop=True, nodata=np.nan)
        clip_profile = src.profile.copy()
        clip_profile.update(height=clipped.shape[1], width=clipped.shape[2], transform=transform)
    with rasterio.open(lst_out, 'w', **clip_profile) as dst:
        dst.write(clipped)
        dst.set_band_description(1, 'LST_predicted_C')

    ndvi_out = os.path.join(out_dir, f'NDVI_{name}_{date_str}.tif')
    with rasterio.open(ndvi_tif_path) as src:
        aoi_in_raster_crs_n = reproject_geom(aoi_local_geom, src.crs)
        clipped_n, transform_n = rio_mask(src, [aoi_in_raster_crs_n.__geo_interface__], crop=True, nodata=np.nan)
        ndvi_profile = src.profile.copy()
        ndvi_profile.update(height=clipped_n.shape[1], width=clipped_n.shape[2],
                             transform=transform_n, dtype='float32', nodata=np.nan)
    with rasterio.open(ndvi_out, 'w', **ndvi_profile) as dst:
        dst.write(clipped_n.astype('float32'))
        dst.set_band_description(1, 'NDVI')

    print(f"Wrote {lst_out}")
    print(f"Wrote {ndvi_out}")
    return lst_out, ndvi_out


# ─────────────────────────────────────────────────────────────────────────
# STEP 3 — run this after CONFIG + DISTRICT are set
# ─────────────────────────────────────────────────────────────────────────

def run_prediction():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    ee = init_ee()
    aoi_geom, aoi_local = load_aoi(ee)

    model = joblib.load(MODEL_PATH)
    meta = joblib.load(META_PATH)

    if DISTRICT is None:
        print("DISTRICT is not set — call print_available_districts(), pick the "
              "matching name, and set DISTRICT in CONFIG before running this.\n"
              "Continuing UNCALIBRATED (bias correction = 0, elevation = AOI's own SRTM value).")
        bias_correction = 0.0
        district_elev_value = None   # resolved to raw elevation per-pixel below
    elif DISTRICT not in meta['district_bias_correction']:
        print(f"WARNING: '{DISTRICT}' not found in the model's calibrated districts. "
              f"Run print_available_districts() to see valid names. "
              f"Continuing UNCALIBRATED.")
        bias_correction = 0.0
        district_elev_value = None
    else:
        bias_correction = meta['district_bias_correction'][DISTRICT]
        district_elev_value = meta['district_elev_map'][DISTRICT]
        print(f"Calibrating for '{DISTRICT}': bias correction = {bias_correction:+.3f}°C, "
              f"district_mean_elevation = {district_elev_value:.1f} m")

    # If no district match, fall back to the AOI's own mean SRTM elevation
    # (same fallback predict_lst() uses when district=None).
    if district_elev_value is None:
        import ee as _ee
        elev_img = _ee.Image(SRTM_ASSET).select('elevation')
        district_elev_value = elev_img.reduceRegion(
            reducer=_ee.Reducer.mean(), geometry=aoi_geom, scale=30, maxPixels=1e9
        ).get('elevation').getInfo()

    stack, ndvi_img = build_feature_image(ee, aoi_geom, TARGET_DATE, district_elev_value)

    stack_tif = os.path.join(OUTPUT_DIR, f'_stack_{AOI_NAME}_{TARGET_DATE}.tif')
    ndvi_tif = os.path.join(OUTPUT_DIR, f'_ndvi_raw_{AOI_NAME}_{TARGET_DATE}.tif')
    export_image_locally(stack, aoi_geom, stack_tif, FEATURE_BAND_ORDER)
    export_image_locally(ndvi_img, aoi_geom, ndvi_tif, ['ndvi'])

    return run_inference(stack_tif, ndvi_tif, meta, model, bias_correction,
                          aoi_local, OUTPUT_DIR, AOI_NAME, TARGET_DATE)


# ─────────────────────────────────────────────────────────────────────────
print_available_districts()   # copy exact name into DISTRICT
run_prediction()